Open in colab: https://colab.research.google.com/github/Thanaritt-K/Thai-BPE-experiment/blob/main/BPEexperiment.ipynb

First, we need to install some dependencies used in this experiment

In [162]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install tltk

The data can be downloaded from https://github.com/korakot/corpus/tree/main/BEST or https://huggingface.co/datasets/nectec/best2009 But for simplicity I'll use the first one, since it's more easily managed. It has already been uploaded to this repo as well, for archival purposes.

In [163]:
import os
import numpy as np
import pandas as pd

# For quick experimentation, we will include only 200 texts
folderpath = r"BEST/TrainingSet/article/" 
files = os.listdir(folderpath)
files = np.sort(files)

files_txt = [i for i in files if i.endswith('.txt')]
training_data = []

for file_name in files_txt:
    text_file = open(folderpath+file_name,'r', encoding='utf8', errors='ignore')
    training_data.append(text_file.readlines())

# flatten the data structure
training_data = [
    x
    for xs in training_data
    for x in xs
]

An example of Thai sentences, with word boundaries annotation from BEST-2010 Corpus and number of sentences we will use to train the BPE segmentor

In [164]:
print(training_data[:5])
len(training_data)

['กฎหมาย|กับ|การ|เบียดบัง|คน|จน|\n', 'จาก|ต้นฉบับ|เรื่อง| |"|บท|นำ|:| |คน|จน|ภาย|ใต้|ความ|สัมพันธ์|ทาง|กฎหมาย|"|\n', '<NE>ไพสิฐ พาณิชย์กุล</NE>|\n', 'อาจารย์|ประจำ| |<NE>สาขาวิชานิติศาสตร์</NE>| |<NE>ภาควิชารัฐศาสตร์</NE>| |<NE>คณะสังคมศาสตร์</NE>| |<NE>มหาวิทยาลัยเชียงใหม่</NE>|\n', '(|บทความ|นี้|ยาว|ประมาณ| |9| |หน้า|กระดาษ| |A|4|)|\n']


16990

Next, we substitues whitespaces with a meta symbol _ (U+2581), remove tags and some punctuation marks, such as quotation marks and parentheses, and then substitues word boundaries (|) with whitespace.

In [165]:
import re

i = 0
training_data_processed = []
while i < len(training_data):
    text = training_data[i]
    text = re.sub(r'</?(AB|NE)>', '', text) #remove tags
    text = re.sub(r'[\"\:\(\)]', '', text) #remove some punctuation marks
    text = re.sub(r' ', '_', text) #substitute whitespaces to preserve them
    text = re.sub(r'\|', ' ', text) #substitute word boundaries to prepare the data for BPE training
    text = re.sub(r' {2,}', ' ', text) #remove double whitespace, but keeping newlines
    training_data_processed.append(text)
    i = i + 1
    

An example of Thai sentences after pre-processing

In [166]:
print(training_data_processed[:5])

['กฎหมาย กับ การ เบียดบัง คน จน \n', 'จาก ต้นฉบับ เรื่อง _ บท นำ _ คน จน ภาย ใต้ ความ สัมพันธ์ ทาง กฎหมาย \n', 'ไพสิฐ_พาณิชย์กุล \n', 'อาจารย์ ประจำ _ สาขาวิชานิติศาสตร์ _ ภาควิชารัฐศาสตร์ _ คณะสังคมศาสตร์ _ มหาวิทยาลัยเชียงใหม่ \n', ' บทความ นี้ ยาว ประมาณ _ 9 _ หน้า กระดาษ _ A 4 \n']


Here we start to write our BPE training algorithm. Most of these are based on Andrej Karpathy's GitHub (https://github.com/karpathy/minbpe).

In [167]:
# This module provides access to the Unicode Character Database (UCD) which defines character properties for all Unicode characters.
import unicodedata

# This function returns a dictionary with consecutive pairs of integers and their counts in the given list of integers.
def get_stats(ids, stats=None):

    stats = {} if stats is None else stats
    # zip function allows to iterate consecutive items from given two list
    for pair in zip(ids, ids[1:]):
        # If a pair already exists in the stats dictionary, add 1 to its value else assign the value as 0.
        stats[pair] = stats.get(pair, 0) + 1
    return stats

# Once we find out the list of consecutive pairs of integers, we'll then replace those pairs with new integer tokens.
def merge(ids, pair, idx):
    newids = []
    i = 0
    # As we'll be merging a pair of ids, hence the minimum id in the list should be 2 or more.
    while i < len(ids):
        # If the current id and next id(id+1) exist in the given pair, and the position of id is not the last, then replace the 2 consecutive id with the given index value.
        if ids[i] == pair[0] and i < len(ids) - 1 and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2  # If the pair is matched, the next iteration starts after 2 positions in the list.
        else:
            newids.append(ids[i])
            i += 1  # Since the current id pair didn't match, so start iteration from the 1 position next in the list.
    # Returns the Merged Ids list
    return newids

# This function checks that using 'unicodedata.category' which returns "C" as the first letter if it is a control character and we'll have to replace it readable character.
def replace_control_characters(s: str) -> str:
    chars = []
    for ch in s:
        # If the character is not distorted (meaning the first letter doesn't start with "C"), then append the character to chars list.
        if unicodedata.category(ch)[0] != "C":
            chars.append(ch)
        # If the character is distorted (meaning the first letter has the letter "C"), then replace it with readable bytes and append to chars list.
        else:
            chars.append(f"\\u{ord(ch):04x}")
    return "".join(chars)

# Some of the tokens such as control characters like Escape Characters can't be decoded into valid strings.
# Hence those need to be replace with readable character such as �
def render_token(t: bytes) -> str:
    s = t.decode('utf-8', errors='replace')
    s = replace_control_characters(s)
    return s

In [168]:
# Create a Thai Tokenizer class.
class ThaiTokenizer():

  def __init__(self):

        # The byte pair should be done within the related words or sentences that give a proper context. Pairing between unrelated words or sentences may give undesirable output.
        self.pattern = r"[A-Z0-9_\\nก-๙}]+"
        self.compiled_pattern = re.compile(self.pattern)

        # Special tokens are used to provide coherence in the sequence while training.
        # Special tokens are assigned a unique index number and stored in vocabulary.
        self.special_tokens = {
            '<|begin_of_text|>': 1101,
            '<|end_of_text|>': 1102,
            '<|start_header_id|>': 1103,
            '<|end_header_id|>': 1104,
            '<|eot_id|>': 1105
        }

        # Initialize merges with empty dictionary
        self.merges = {}

        # Initialize the vocab dictionary by calling the function _build_vocab which is defined later in this class.
        self.vocab = self._build_vocab()

  # Tokenizer training function
  def train(self, text, vocab_size):

        # Make sure the vocab size must be at least 256 as the utf-8 encoding for the range 0-255 are same as the Ascii character.
        assert vocab_size >= 256
        # Total number of merges into the vocabulary.
        num_merges = vocab_size - 256

        # The first step is to make sure to split the text up into text chunks using the pattern defined above.
        text_chunks = re.findall(self.compiled_pattern, str(text))

        # Each text_chunks will be utf-8 encoded to bytes and then converted into an integer list.
        ids = [list(ch.encode("utf-8")) for ch in text_chunks]

        # Iteratively merge the most common pairs to create new tokens
        merges = {} # (int, int) -> int
        vocab = {idx: bytes([idx]) for idx in range(256)} # idx -> bytes

        # Until the total num_merges is reached, find the common pair of consecutive id in the ids list and start merging them to create a new token
        for i in range(num_merges):
            # Count the number of times every consecutive pair appears
            stats = {}
            for chunk_ids in ids:
                # Passing in stats will update it in place, adding up counts
                get_stats(chunk_ids, stats)
            # Find the pair with the highest count
            pair = max(stats, key=stats.get)
            # Mint a new token: assign it the next available id
            idx = 256 + i
            # Replace all occurrences of pair in ids with idx
            ids = [merge(chunk_ids, pair, idx) for chunk_ids in ids]
            # Save the merge
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

        # Save class variables to be used later during tokenizer encode and decode
        self.merges = merges
        self.vocab = vocab

  # Tokenizer decode function takes a list of integer ids and return strings
  def decode(self, ids):

        # Initialize empty byte list
        part_bytes = []
        # Change the position of keys and values of special_tokens dict and store into inverse_special_tokens
        inverse_special_tokens = {v: k for k, v in self.special_tokens.items()}

        # Loop through idx in the ids list
        for idx in ids:
            # If the idx is found in vocab dict, get the bytes of idx and append them into part_bytes list
            if idx in self.vocab:
                part_bytes.append(self.vocab[idx])
            # If the idx is found in inverse_special_tokens dict, get the token string of the corresponding idx, convert it to bytes using utf-8 encode and then append it into part_bytes list
            elif idx in inverse_special_tokens:
                part_bytes.append(inverse_special_tokens[idx].encode("utf-8"))
            # If the idx is not found in both vocab and special token dict, throw an invalid error
            else:
                raise ValueError(f"invalid token id: {idx}")

        # Join all the individual bytes from the part_byte list
        text_bytes = b"".join(part_bytes)

        # Convert the bytes to text string using utf-8 decode function. Make sure to use "errors=replace" to replace distorted characters with readable characters such as �.
        text = text_bytes.decode("utf-8", errors="replace")
        return text

  # Tokenizer encode function takes text as a string and returns integer ids list
  def encode(self, text):

        # Define a pattern to identify special token present in the text
        special_pattern = "(" + "|".join(re.escape(k) for k in self.special_tokens) + ")"
        # Split special token (if present) from the rest of the text
        special_chunks = re.split(special_pattern, text)
        # Initialize empty ids list
        ids = []

        # Loop through each of parts in the special chunks list.
        for part in special_chunks:
            # If the part of the text is the special token, get the idx of the part from the special token dictionary and append it to the ids list.
            if part in self.special_tokens:
                ids.append(self.special_tokens[part])
            # If the part of text is not a special token
            else:
                # Split the text into multiple chunks using the pattern we've defined earlier.
                text_chunks = re.findall(self.compiled_pattern, str(text))

                # All text chunks are encoded separately, then the results are joined
                for chunk in text_chunks:
                    chunk_bytes = chunk.encode("utf-8")   # Encode text to bytes
                    chunk_ids = list(chunk_bytes)         # Convert bytes to list of integer

                    while len(chunk_ids) >= 2:    # chunks ids list must be at least 2 id to form a byte-pair
                        # Count the number of times every consecutive pair appears
                        stats = get_stats(chunk_ids)
                        # Some idx pair might be created with another idx in the merge dictionary. Hence we'll find the pair with the lowest merge index to ensure we cover all byte pairs in the merge dict.
                        pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))

                        # Break the loop and return if the pair is not present in the merges dictionary
                        if pair not in self.merges:
                            break
                        # Find the idx of the pair present in the merges dictionary
                        idx = self.merges[pair]
                        # Replace the occurrences of pair in ids list with this idx and continue
                        chunk_ids = merge(chunk_ids, pair, idx)

                    ids.extend(chunk_ids)
        return ids

  # Function to return a vocab dictionary combines with merges and special tokens
  def _build_vocab(self):
        # The utf-8 encoding for the range 0-255 are same as the Ascii character.
        vocab = {idx: bytes([idx]) for idx in range(256)}

        # Iterate through merge dictionary and add into vocab dictionary
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]

        # Iterate through special token dictionary and add into vocab dictionary
        for special, idx in self.special_tokens.items():
            vocab[idx] = special.encode("utf-8")

        return vocab

  # After training is complete, use the save function to save the model file and vocab file.
  # Model file will be used to load the tokenizer model for further use in llm
  # Vocab file is just for the purpose of human verification
  def save(self, file_prefix):
        # Writing to model file
        model_file = file_prefix + ".model"           # model file name

        # Model write begins
        with open(model_file, 'w') as f:
            f.write("thai tokenizer v1.0\n")          # write the tokenizer version
            f.write(f"{len(self.special_tokens)}\n")  # write the length of special tokens

            # Write each special token in the specific format like below
            for tokens, idx in self.special_tokens.items():
                f.write(f"{tokens} {idx}\n")

            # Write only the keys part from the merges dict
            for idx1, idx2 in self.merges:
                f.write(f"{idx1} {idx2}\n")

        # Writing to the vocab file
        vocab_file = file_prefix + ".vocab"       # vocab file name

        # Change the position of keys and values of merge dict and store into inverted_merges
        inverted_merges = {idx: pair for pair, idx in self.merges.items()}
        # Vocab write begins
        with open(vocab_file, "w", encoding="utf-8") as f:
            for idx, token in self.vocab.items():
                # render_token function processes tokens and prevents distorted bytes by replacing them with readable character
                s = render_token(token)
                # If the index of vocab is present in merge dict, then find its child index, convert their corresponding bytes in vocab dict and write the characters
                if idx in inverted_merges:
                    idx0, idx1 = inverted_merges[idx]
                    s0 = render_token(self.vocab[idx0])
                    s1 = render_token(self.vocab[idx1])
                    f.write(f"[{s0}][{s1}] -> [{s}] {idx}\n")
                # If index of vocab is not present in merge dict, just write it's index and the corresponding string
                else:
                    f.write(f"[{s}] {idx}\n")

  # Function to load tokenizer model.
  # This function is invoked only after the training is complete and the tokenizer model file is saved.
  def load(self, model_file):

        merges = {}             # Initialize merge and special_tokens with empty dict
        special_tokens = {}     # Initialize special_tokens with empty dict
        idx = 256               # As the range (0, 255) is already reserved in vocab. So the next index only starts from 256 and onwards.

        # Read model file
        with open(model_file, 'r', encoding="utf-8") as f:

            version = f.readline().strip()          # Read the tokenizer version as defined during model file writing
            num_special = int(f.readline().strip()) # Read the length of special tokens

            # Read all the special tokens and store in special_tokens dict defined earlier
            for _ in range(num_special):
                special, special_idx = f.readline().strip().split()
                special_tokens[special] = int(special_idx)

            # Read all the merge indexes from the file. Make it a key pair and store it in merge dictionary defined earlier.
            # The value of this key pair would be idx(256) as defined above and keep on increase by 1.
            for line in f:
                idx1, idx2 = map(int, line.split())
                merges[(idx1, idx2)] = idx
                idx += 1

        self.merges = merges
        self.special_tokens = special_tokens

        # Create a final vocabulary dictionary by combining merge, special_token and vocab (0-255). _build_vocab function helps to do just that.
        self.vocab = self._build_vocab()

In [ ]:
# Train the tokenizer

import time   # To caculate the duration of training completion

# For testing the experiment
#training_data_processed = training_data_processed[0:1000]

# Define vocab size
vocab_size = 500
# Initialize a tokenizer model class
tokenizer = ThaiTokenizer()
# Start train a tokenizer
start_time = time.time()
tokenizer.train(training_data_processed, vocab_size)
end_time = time.time()
# Save tokenizer: you can change path and filename.
tokenizer.save("models/thaitokenizer_500")
print(f"Total time to complete tokenizer training: {end_time-start_time:.2f} seconds")

Total time to complete tokenizer training: 359.21 seconds


In [305]:
# Test the tokenizer

# Initialize a tokenizer model class
tokenizer = ThaiTokenizer()
# Load tokenizer model. This model was saved during training.
tokenizer.load("./models/thaitokenizer_500.model")

# Invoke and verify the tokenizer encode and decode function for Thai Language
thai_texts = "คนจนภายใต้ความสัมพันธ์ทางกฎหมาย"
print(f"Thai Text: {thai_texts}")
thai_encoded_ids = tokenizer.encode(thai_texts)
print(f"Encoded Ids: {thai_encoded_ids}")
#thai_decoded_texts = tokenizer.decode(thai_encoded_ids)
#print(f"Decoded Texts: {thai_decoded_texts}")
print(f"Vocab:     ", end="")
for thai_encoded_id in thai_encoded_ids:
    print((tokenizer.vocab[thai_encoded_id]).decode("utf-8", errors="replace"), end=" ")
print(f"\nMorphemes: คน จน ภาย ใต้ ความ สัมพันธ์ ทาง กฏ หมาย")

Thai Text: คนจนภายใต้ความสัมพันธ์ทางกฎหมาย
Encoded Ids: [406, 287, 260, 344, 306, 286, 278, 267, 325, 282, 266, 269, 300, 321, 329, 305, 368, 494, 441]
Vocab:     คน จ น ภ าย ใ ต ้ ความ ส ั ม พ ัน ธ ์ ทาง กฎ หมาย 
Morphemes: คน จน ภาย ใต้ ความ สัมพันธ์ ทาง กฏ หมาย


Then we can compare the BPE result with test result from a syllable-based tokenizer.

In [306]:
import tltk

tltk.nlp.syl_segment(thai_texts)

'คน~จน~ภาย~ใต้~ความ~สัม~พันธ์~ทาง~กฎ~หมาย~<s/>'

Next we can test the tokenizer on an unseen sentence not included in the training dataset.

In [308]:
thai_texts = "การเมืองภาคประชาชน คือ การเคลื่อนไหวของประชาชน"
print(f"Thai Text: {thai_texts}")
thai_encoded_ids = tokenizer.encode(thai_texts)
print(f"Encoded Ids: {thai_encoded_ids}")
#thai_decoded_texts = tokenizer.decode(thai_encoded_ids)
#print(f"Decoded Texts: {thai_decoded_texts}")
print(f"Vocab:     ", end="")
for thai_encoded_id in thai_encoded_ids:
    print((tokenizer.vocab[thai_encoded_id]).decode("utf-8", errors="replace"), end=" ")
print(f"\nMorphemes: การ เมือง ภาค ประชา ชน คือ การ เคลื่อน ไหว ของ ประชา ชน")

Thai Text: การเมืองภาคประชาชน คือ การเคลื่อนไหวของประชาชน
Encoded Ids: [292, 479, 395, 132, 335, 420, 138, 260, 463, 292, 486, 276, 352, 260, 290, 283, 272, 315, 335, 420, 138, 260]
Vocab:     การ เมือง ภา� � ประ ชา� � น คือ การ เค ล ื่อ น ไ ห ว ของ ประ ชา� � น 
Morphemes: การ เมือง ภาค ประชา ชน คือ การ เคลื่อน ไหว ของ ประชา ชน


In [310]:
tltk.nlp.syl_segment(thai_texts)

'การ~เมือง~ภาค~ประ~ชา~ชน~<s/>คือ~ ~การ~เคลื่อน~ไหว~ของ~ประ~ชา~ชน~<s/>'

Then we experiment with bigger vocabulary size: 3000 and 5000 to see if this improves BPE results.

In [ ]:
vocab_size = 3000
start_time = time.time()
tokenizer.train(training_data_processed, vocab_size)
end_time = time.time()
tokenizer.save("models/thaitokenizer_3000")
print(f"Total time to complete tokenizer training: {end_time-start_time:.2f} seconds")

In [ ]:
vocab_size = 5000
start_time = time.time()
tokenizer.train(training_data_processed, vocab_size)
end_time = time.time()
tokenizer.save("models/thaitokenizer_5000")
print(f"Total time to complete tokenizer training: {end_time-start_time:.2f} seconds")

In [311]:
thai_texts = "คนจนภายใต้ความสัมพันธ์ทางกฎหมาย"
print(f"Thai Text: {thai_texts}")
print(f"\nMorphemes:     คน จน ภาย ใต้ ความ สัมพันธ์ ทาง กฏ หมาย")

tokenizer.load("./models/thaitokenizer_500.model")
thai_encoded_ids = tokenizer.encode(thai_texts)
print(f"Vocab(500):    ", end="")
for thai_encoded_id in thai_encoded_ids:
    print((tokenizer.vocab[thai_encoded_id]).decode("utf-8", errors="replace"), end=" ")

tokenizer.load("./models/thaitokenizer_3000.model")
thai_encoded_ids = tokenizer.encode(thai_texts)
print(f"\nVocab(3000):   ", end="")
for thai_encoded_id in thai_encoded_ids:
    print((tokenizer.vocab[thai_encoded_id]).decode("utf-8", errors="replace"), end=" ")

tokenizer.load("./models/thaitokenizer_5000.model")
thai_encoded_ids = tokenizer.encode(thai_texts)
print(f"\nVocab(5000):   ", end="")
for thai_encoded_id in thai_encoded_ids:
    print((tokenizer.vocab[thai_encoded_id]).decode("utf-8", errors="replace"), end=" ")
print("\nSyllable-based:" + re.sub("~|<s/>"," ",tltk.nlp.syl_segment(thai_texts)))

Thai Text: คนจนภายใต้ความสัมพันธ์ทางกฎหมาย

Morphemes:     คน จน ภาย ใต้ ความ สัมพันธ์ ทาง กฏ หมาย
Vocab(500):    คน จ น ภ าย ใ ต ้ ความ ส ั ม พ ัน ธ ์ ทาง กฎ หมาย 
Vocab(3000):   คน จน ภาย ใต้ ความ สัมพันธ์ ทาง กฎหมาย 
Vocab(5000):   คน จน ภาย ใต้ ความ สัมพันธ์ ทาง กฎหมาย 
Syllable-based:คน จน ภาย ใต้ ความ สัม พันธ์ ทาง กฎ หมาย  


In [313]:
thai_texts = "การเมืองภาคประชาชน คือ การเคลื่อนไหวของประชาชน"
print(f"Thai Text: {thai_texts}")
print(f"\nMorphemes:     การ เมือง ภาค ประชา ชน คือ การ เคลื่อน ไหว ของ ประชา ชน")

tokenizer.load("./models/thaitokenizer_500.model")
thai_encoded_ids = tokenizer.encode(thai_texts)
print(f"Vocab(500):    ", end="")
for thai_encoded_id in thai_encoded_ids:
    print((tokenizer.vocab[thai_encoded_id]).decode("utf-8", errors="replace"), end=" ")

tokenizer.load("./models/thaitokenizer_3000.model")
thai_encoded_ids = tokenizer.encode(thai_texts)
print(f"\nVocab(3000):   ", end="")
for thai_encoded_id in thai_encoded_ids:
    print((tokenizer.vocab[thai_encoded_id]).decode("utf-8", errors="replace"), end=" ")

tokenizer.load("./models/thaitokenizer_5000.model")
thai_encoded_ids = tokenizer.encode(thai_texts)
print(f"\nVocab(5000):   ", end="")
for thai_encoded_id in thai_encoded_ids:
    print((tokenizer.vocab[thai_encoded_id]).decode("utf-8", errors="replace"), end=" ")

print("\nSyllable-based:" + re.sub("~|<s/>"," ",tltk.nlp.syl_segment(thai_texts)))

Thai Text: การเมืองภาคประชาชน คือ การเคลื่อนไหวของประชาชน

Morphemes:     การ เมือง ภาค ประชา ชน คือ การ เคลื่อน ไหว ของ ประชา ชน
Vocab(500):    การ เมือง ภา� � ประ ชา� � น คือ การ เค ล ื่อ น ไ ห ว ของ ประ ชา� � น 
Vocab(3000):   การ เมือง ภาค ประชาชน คือ การ เคลื่อนไหว ของ ประชาชน 
Vocab(5000):   การเมือง ภาค ประชาชน คือ การ เคลื่อนไหว ของ ประชาชน 
Syllable-based:การ เมือง ภาค ประ ชา ชน  คือ   การ เคลื่อน ไหว ของ ประ ชา ชน  
